In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import zipfile

In [6]:
src = Path("byggnader_sverige")
dst = Path("byggnader_unzipped")
dst.mkdir(exist_ok=True)

for z in sorted(src.glob("*.zip")):
    with zipfile.ZipFile(z) as zf:
        zf.extractall(dst)
    z.unlink()   # delete the zip after it was extracted

print(len(list(dst.rglob("*.gpkg"))), "gpkg files")

290 gpkg files


In [2]:
files = sorted(Path("byggnader_unzipped").glob("*.gpkg"))
gdf = pd.concat((gpd.read_file(f) for f in files), ignore_index=True)
gdf = gpd.GeoDataFrame(gdf, crs=gdf.crs)

print(len(gdf), "buildings")
gdf.head()

9524202 buildings


,objektidentitet,versiongiltigfran,lagesosakerhetplan,lagesosakerhethojd,ursprunglig_organisation,objektversion,objekttypnr,objekttyp,insamlingslage,byggnadsnamn1,byggnadsnamn2,byggnadsnamn3,husnummer,huvudbyggnad,andamal1,andamal2,andamal3,andamal4,andamal5,geometry
0,201ed2b1-bf28-44ed-aada-14919eef0fda,2023-01-09 14:56:46.951000+00:00,0.20,0.5,Upplands Väsby,2,2066,Komplementbyggnad,Takkant,NaN,NaN,None,79.0,Nej,Komplementbyggnad;,NaN,NaN,None,None,"MULTIPOLYGON (((669286.32 6596243.91, 669287.2..."
1,0232b918-1c77-47db-b085-b274b42c44b1,2023-01-10 09:39:03.301000+00:00,0.20,0.5,Upplands Väsby,2,2066,Komplementbyggnad,Fasad,NaN,NaN,None,12.0,Nej,Komplementbyggnad;,NaN,NaN,None,None,"MULTIPOLYGON (((665789.684 6601004.11, 665789...."
2,8438371a-80a1-4d5d-9690-859525b2312d,2023-04-25 16:00:13.443000+00:00,0.25,0.5,Upplands Väsby,1,2066,Komplementbyggnad,Takkant,NaN,NaN,None,19.0,Nej,Komplementbyggnad;,NaN,NaN,None,None,"MULTIPOLYGON (((667570.45 6601707.884, 667567...."
3,65c2fe92-7a2d-4312-921a-21f1624bd47c,2023-06-21 10:38:31.239000+00:00,0.05,0.5,Upplands Väsby,2,2067,Övrig byggnad,Fasad,NaN,NaN,None,6.0,Nej,Övrig byggnad;,NaN,NaN,None,None,"MULTIPOLYGON (((661938.681 6600301.496, 661936..."
4,3e2fe683-07a4-4148-8549-a6d76f9120f1,2024-01-25 10:23:15.505000+00:00,0.03,0.5,Upplands Väsby,3,2066,Komplementbyggnad,Fasad,NaN,NaN,None,3.0,Nej,Komplementbyggnad;,NaN,NaN,None,None,"MULTIPOLYGON (((666513.128 6605628.705, 666512..."


In [3]:
gdf=gdf.to_crs(3035)

In [11]:
gdf["area_m2"] = gdf.geometry.area
top100 = gdf.nlargest(100, "area_m2")

In [5]:
gdf.to_parquet("/data/uscuni-ulce/extension2/swe/clean_3035.parquet")